In [1]:
import pandas as pd

In [2]:
# IPEDS 2011 data

# c2011 = pd.read_csv(r"C:\capstone_data\raw\c2011_a.csv")

In [33]:
# Define columns needed and create dataframes from IPEDS 2011 and 2021

cols = ["UNITID", "CIPCODE", "AWLEVEL", "CTOTALT", "CTOTALM", "CTOTALW"]
c2011_clean = pd.read_csv(r"C:\capstone_data\raw\c2011_a.csv", usecols=cols)
c2021_clean = pd.read_csv(r"C:\capstone_data\raw\c2021_a.csv", usecols=cols)

In [34]:
# Dictionary for renaming columns

rename_dict = {
    "UNITID" : "unitid"
    ,"CIPCODE" : "cipcode"
    ,"AWLEVEL" : "credlev"
    ,"CTOTALT" : "completions"
    ,"CTOTALM" : "completions_men"
    ,"CTOTALW" : "completions_women"
}

In [35]:
# Rename the columns

c2011_clean = c2011_clean.rename(columns=rename_dict)
c2021_clean = c2021_clean.rename(columns=rename_dict)

In [36]:
print(c2011_clean.columns == c2021_clean.columns)

[ True  True  True  True  True  True]


In [37]:
# Set years

c2011_clean["year"] = 2011
c2021_clean["year"] = 2021

In [38]:
completions = pd.concat([c2011_clean, c2021_clean], ignore_index=True)

In [15]:
c2011_clean["credlev"].value_counts().sort_index()
c2021_clean["credlev"].value_counts().sort_index()

credlev
2      30085
3      48506
4       2578
5     103820
6      12450
7      43913
8       5179
17     13130
18      2866
19       383
20      3762
21     29671
Name: count, dtype: int64

In [16]:
# Load College Scorecard Field of Study data

scorecard_fos = pd.read_csv(r"C:\capstone_data\raw\Most-Recent-Cohorts-Field-of-Study.csv")

In [17]:
scorecard_fos.shape

(229188, 174)

In [18]:
scorecard_fos.columns

Index(['UNITID', 'OPEID6', 'INSTNM', 'CONTROL', 'MAIN', 'CIPCODE', 'CIPDESC',
       'CREDLEV', 'CREDDESC', 'IPEDSCOUNT1',
       ...
       'EARN_COUNT_PELL_WNE_5YR', 'EARN_PELL_WNE_MDN_5YR',
       'EARN_COUNT_NOPELL_WNE_5YR', 'EARN_NOPELL_WNE_MDN_5YR',
       'EARN_COUNT_MALE_WNE_5YR', 'EARN_MALE_WNE_MDN_5YR',
       'EARN_COUNT_NOMALE_WNE_5YR', 'EARN_NOMALE_WNE_MDN_5YR',
       'EARN_COUNT_HIGH_CRED_5YR', 'EARN_IN_STATE_5YR'],
      dtype='object', length=174)

In [19]:
# Select columns from scorecard to keep. EARN_NE_MDN_3YR provides median earnings 3 years after completion for non-enrolled students.

scorecard_fos_clean = scorecard_fos[[
    "UNITID"
    ,"INSTNM"
    ,"CIPCODE"
    ,"CREDLEV"
    ,"EARN_NE_MDN_3YR"
]].copy()

In [20]:
scorecard_fos_clean.rename(columns={
    "UNITID" : "unitid"
    ,"INSTNM" : "inst_name"
    ,"CIPCODE" : "cipcode"
    ,"CREDLEV" : "credlev"
    ,"EARN_NE_MDN_3YR" : "median_earnings_3yr"
})

,unitid,inst_name,cipcode,credlev,median_earnings_3yr
0,100654.0,Alabama A & M University,100,3,PS
1,100654.0,Alabama A & M University,101,3,NaN
2,100654.0,Alabama A & M University,109,3,PS
3,100654.0,Alabama A & M University,110,3,PS
4,100654.0,Alabama A & M University,110,5,PS
...,...,...,...,...,...
229183,NaN,Southeast New Mexico College,5201,2,NaN
229184,NaN,Southeast New Mexico College,5203,1,NaN
229185,NaN,Southeast New Mexico College,5204,1,NaN
229186,NaN,Southeast New Mexico College,5204,2,NaN


In [21]:
scorecard_fos_clean.head(2)

,UNITID,INSTNM,CIPCODE,CREDLEV,EARN_NE_MDN_3YR
0,100654.0,Alabama A & M University,100,3,PS
1,100654.0,Alabama A & M University,101,3,NaN


In [22]:
# Load College Scorecard Institution Level data

scorecard_il = pd.read_csv(r"C:\capstone_data\raw\Most-Recent-Cohorts-Institution.csv", usecols=["UNITID", "INSTNM", "STABBR"])

In [23]:
scorecard_il.head(2)

,UNITID,INSTNM,STABBR
0,100654,Alabama A & M University,AL
1,100663,University of Alabama at Birmingham,AL


In [24]:
scorecard_fos_clean.columns

Index(['UNITID', 'INSTNM', 'CIPCODE', 'CREDLEV', 'EARN_NE_MDN_3YR'], dtype='object')

In [25]:
scorecard_il["STABBR"].value_counts().head()

STABBR
CA    671
NY    428
TX    415
FL    393
PA    331
Name: count, dtype: int64

In [26]:
inst_clean = scorecard_il.rename(columns={
    "UNITID" : "unitid"
    ,"INSTNM" : "inst_name"
    ,"STABBR" : "state_abbr"
})

In [27]:
completions_full = completions.merge(
    inst_clean,
    on="unitid"
    ,how="left"
)

In [28]:
completions_tn = completions_full[completions_full["state_abbr"] == "TN"]

In [29]:
ic2011 = pd.read_csv(r"C:\capstone_data\raw\IC2011_AY.csv")
print(ic2011.head())

       UNITID XTUIT1 TUITION1 XFEE1 FEE1 XHRCHG1 HRCHG1 XCMPFEE1 CMPFEE1  \
100636      R      0        R     0    R       0      A        .       R   
100654      R   5328        R  1500    R     222      A        .       R   
100663      R   6264        R     0    R     246      A        .       R   
100690      R   7920        R   800    R     330      A        .       R   
100706      R   8094        R     0    R     257      A        .       R   

       XTUIT2  ... CMP2GTD XCMP3AY0 CMP3AY0 XCMP3AY1 CMP3AY1 XCMP3AY2 CMP3AY2  \
100636      0  ...       A        .       A        .       A        .       A   
100654   5328  ...       A        .       A        .       A        .       A   
100663   6264  ...       A        .       A        .       A        .       A   
100690   7920  ...       A        .       A        .       A        .       A   
100706   8094  ...       A        .       A        .       A        .       A   

       XCMP3AY3 CMP3AY3 CMP3GTD   
100636        .      

In [30]:
print(ic2011.shape)

(4793, 268)


In [31]:
print(ic2011.columns)

Index(['UNITID', 'XTUIT1', 'TUITION1', 'XFEE1', 'FEE1', 'XHRCHG1', 'HRCHG1',
       'XCMPFEE1', 'CMPFEE1', 'XTUIT2',
       ...
       'CMP2GTD', 'XCMP3AY0', 'CMP3AY0', 'XCMP3AY1', 'CMP3AY1', 'XCMP3AY2',
       'CMP3AY2', 'XCMP3AY3', 'CMP3AY3', 'CMP3GTD '],
      dtype='object', length=268)
